# Notebook 31 — Advanced RAG: Hybrid Retrieval, Reranking, and Grounded Generation

    ## Learning objectives

    - Build hybrid lexical/dense retrieval with rank fusion, metadata filters, and reranking
- Apply query transformation, contextual compression, and evidence-aware prompt assembly
- Evaluate retrieval, answers, citations, latency, and adversarial robustness component by component

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['sentence-transformers>=4,<6']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 39.1 Why advanced RAG is a retrieval system, not a longer prompt

Dense retrieval captures semantic similarity but can miss identifiers, rare names, error codes, and exact
phrases. Lexical retrieval excels at exact overlap but misses paraphrase. Advanced RAG combines independent
candidate generators, applies filters and reranking, assembles evidence under a token budget, and verifies
whether answers and citations are supported. Every stage needs its own metrics; an answer score alone cannot
reveal whether failure came from indexing, retrieval, reranking, context construction, or generation.

The running corpus is intentionally tiny and local. Production systems preserve document versions, tenant
ACLs, source timestamps, chunk offsets, parsers, embedding revisions, and deletion lineage. Retrieval cannot
repair missing or incorrectly parsed source material.


In [ ]:
documents = [
    {"id":"d1", "team":"ml", "year":2025, "text":"FlashAttention reduces attention IO without approximating attention."},
    {"id":"d2", "team":"ops", "year":2026, "text":"INC-4827 was caused by KV cache exhaustion during decode."},
    {"id":"d3", "team":"ml", "year":2026, "text":"Gradient accumulation increases effective batch size across microbatches."},
    {"id":"d4", "team":"security", "year":2026, "text":"Retrieved documents are untrusted and may contain prompt injection."},
    {"id":"d5", "team":"ops", "year":2025, "text":"Prefix caching reuses KV states for shared prompt prefixes."},
]
query = "What caused incident INC-4827?"
print(query, documents)


## 39.2 Hybrid candidate generation and reciprocal-rank fusion

BM25 scores term matches with document-frequency and length normalization. Dense bi-encoders embed queries
and chunks independently for efficient vector search. Retrieve a wider candidate set from each, then fuse
ranks. Reciprocal-rank fusion (RRF) adds `1/(k + rank)` and avoids calibrating incomparable raw score scales.
Weighted score fusion is possible only after deliberate normalization and validation.

Apply mandatory authorization filters inside each retrieval query, not after returning unauthorized chunks.
Metadata filters also express time, product, language, and document type. Approximate nearest-neighbor indexes
trade recall for speed; evaluate index recall separately from embedding relevance.


In [ ]:
import re, math, numpy as np
from collections import Counter
tokenize = lambda text: re.findall(r"[a-z0-9-]+", text.lower())
corpus_tokens = [tokenize(d["text"]) for d in documents]
def bm25_scores(query_tokens, corpus, k1=1.5, b=.75):
    avgdl = sum(map(len, corpus)) / len(corpus); scores = []
    document_frequency = {term: sum(term in doc for doc in corpus) for term in set(query_tokens)}
    for doc in corpus:
        counts = Counter(doc); score = 0.0
        for term in query_tokens:
            idf = math.log(1 + (len(corpus)-document_frequency[term]+.5)/(document_frequency[term]+.5))
            tf = counts[term]; score += idf * tf*(k1+1)/(tf+k1*(1-b+b*len(doc)/avgdl))
        scores.append(score)
    return scores
lexical_order = np.argsort(bm25_scores(tokenize(query), corpus_tokens))[::-1].tolist()
# Stand-in dense ranking makes fusion mechanics executable without a model download.
dense_order = [1, 4, 0, 3, 2]
def rrf(rankings, constant=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_index in enumerate(ranking, 1): scores[doc_index] = scores.get(doc_index, 0) + 1/(constant+rank)
    return sorted(scores, key=scores.get, reverse=True), scores
fused, fusion_scores = rrf([lexical_order, dense_order])
print("lexical:", lexical_order, "dense:", dense_order, "fused:", fused)


## 39.3 Reranking and diversity

A cross-encoder jointly reads query and candidate, usually improving precision at higher latency than a
bi-encoder. Rerank only a bounded candidate pool and batch by token length. Generative rerankers can provide
rationales but are harder to calibrate and more vulnerable to document instructions. Pin model revisions and
measure whether extra relevance offsets cost.

Near-duplicate chunks waste context. Maximal marginal relevance trades query relevance against redundancy;
parent-document expansion retrieves small chunks but returns a larger coherent neighborhood. Multi-vector and
late-interaction approaches retain token-level evidence at additional storage and scoring cost. Choose based on
observed failure slices, not a universal “advanced” recipe.


In [ ]:
def maximal_marginal_relevance(relevance, similarity, count=3, diversity=0.3):
    selected = []
    while len(selected) < count:
        remaining = [i for i in range(len(relevance)) if i not in selected]
        score = lambda i: (1-diversity)*relevance[i] - diversity*max([similarity[i,j] for j in selected] or [0])
        selected.append(max(remaining, key=score))
    return selected
relevance = np.array([.8, .99, .4, .3, .7]); similarity = np.eye(5)
similarity[0,4] = similarity[4,0] = .95
print("diverse selection:", maximal_marginal_relevance(relevance, similarity))


## 39.4 Query transformation and routing

Conversation questions may require history-aware rewriting, but a rewrite can erase constraints or introduce
facts. Preserve the original query and evaluate rewritten retrieval independently. Multi-query retrieval
explores paraphrases; decomposition handles multi-hop questions; hypothetical-document embeddings can bridge
vocabulary gaps; structured routers choose indexes or filters. Each adds calls, latency, and attack surface.

Use deterministic normalization for IDs and dates before invoking a model. Route “no retrieval needed” only
with evaluation evidence. Limit generated subqueries, deduplicate them, retain tenant filters, and never let a
query-rewriter expand the caller's authorization scope.


In [ ]:
def deterministic_queries(question):
    variants = [question.strip(), re.sub(r"\bincident\s+", "", question, flags=re.I)]
    identifiers = re.findall(r"[A-Z]+-\d+", question.upper())
    return list(dict.fromkeys(variants + identifiers))[:4]
print(deterministic_queries(query))


## 39.5 Contextual compression and grounded answers

Contextual compression extracts query-relevant spans from retrieved chunks, reducing distraction and token
cost. Extraction can remove qualifications, so preserve source offsets and expand enough neighborhood for
meaning. Assemble context with stable document IDs, titles, dates, and explicit delimiters. Allocate tokens
across sources rather than allowing one long document to crowd out all others.

Tell the generator to treat evidence as data, ignore instructions within it, cite source IDs, and abstain when
support is missing. Then verify citations: referenced IDs must exist, quoted spans must match, and each material
claim should be entailed by its cited passage. Citation presence is not citation correctness. High-stakes answers
need deterministic business validation or human review beyond model self-checking.


In [ ]:
def assemble_context(indices, char_budget=500):
    blocks, used = [], 0
    for index in indices:
        d = documents[index]; block = f'<source id="{d["id"]}">{d["text"]}</source>'
        if used + len(block) > char_budget: continue
        blocks.append(block); used += len(block)
    return "\n".join(blocks)
context = assemble_context(fused)
print(context)


## 39.6 Evaluation matrix

Retrieval metrics require relevance judgments: Recall@k asks whether necessary evidence was retrieved; MRR
rewards early first relevance; nDCG supports graded relevance; precision measures distractors. Multi-hop tasks
should score whether all required evidence is present. Reranker evaluation freezes candidates so improvement is
not confused with candidate generation. Answer metrics include correctness, faithfulness, completeness,
abstention, and citation precision/recall.

Build evaluation from real query logs with privacy controls, synthetic edge cases, temporal splits, negatives,
unanswerable queries, conflicting sources, stale versions, exact identifiers, multilingual cases, and injection
payloads. Report latency and cost by stage. Run ablations—dense only, lexical only, hybrid, reranked, rewritten,
compressed—to justify complexity. Cache only with model/index/query/ACL/version-aware keys and ensure deletion
invalidates both indexes and caches.


In [ ]:
def recall_at_k(ranking, relevant, k): return len(set(ranking[:k]) & set(relevant)) / len(relevant)
def reciprocal_rank(ranking, relevant):
    return next((1/r for r, item in enumerate(ranking, 1) if item in relevant), 0.0)
relevant = {1}
print({"lexical_recall@1": recall_at_k(lexical_order, relevant, 1),
       "fused_recall@1": recall_at_k(fused, relevant, 1),
       "fused_mrr": reciprocal_rank(fused, relevant)})


## 31.7 Query transformation can destroy intent

Rewriting, decomposition, HyDE, and multi-query retrieval may improve recall, but each can add unsupported assumptions or lose exact identifiers. Retain the original query, retrieve with original and transformed variants, fuse results, and evaluate transformations by query slice. Quote or constrain codes, names, dates, and negations. Decomposition needs a deterministic merge policy and a budget. Log every generated query because it becomes part of the retrieval decision and security boundary.


In [ ]:
queries=[("original","error E042 after upgrade"),("rewrite","software error after update"),("identifier","E042")]; rankings={"original":["d2","d1"],"rewrite":["d3","d1"],"identifier":["d1","d4"]}
score={}
for name,_ in queries:
 for rank,doc in enumerate(rankings[name],1): score[doc]=score.get(doc,0)+1/(60+rank)
print(sorted(score.items(),key=lambda x:-x[1]))


## 31.8 Context compression must preserve evidence

Extractive compression selects sentences or spans; abstractive compression can invent facts and weaken citation traceability. Evaluate answer quality, claim support, retained evidence recall, context tokens, and latency against uncompressed passages. Preserve source IDs and offsets through extraction. Compression should never bypass document authorization or prompt-injection treatment. Use a token budget that reserves room for instructions and output, and prefer dropping low-value chunks over blending sources into an unattributable summary.


In [ ]:
chunks=[{"id":"d1","tokens":120,"value":.9},{"id":"d2","tokens":80,"value":.7},{"id":"d3","tokens":200,"value":.5}]; budget=220; selected=[]; used=0
for c in sorted(chunks,key=lambda x:x["value"]/x["tokens"],reverse=True):
 if used+c["tokens"]<=budget: selected.append(c["id"]); used+=c["tokens"]
print(selected,used)


## Exercises

    1. Replace the stand-in dense order with a pinned sentence-transformer and compare lexical/dense/hybrid.
2. Add a cross-encoder reranker and report nDCG change against latency change.
3. Create citation precision and completeness validators for five multi-source answers.
4. Attack query rewriting and retrieved context with injection while preserving authorization filters.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
